# 경제 분석 및 예측과 데이터 지능 실습2 (sktime 버전)

이 노트북은 기존 `Prophet/NeuralProphet` 실습을 **sktime 기반 forecasting 파이프라인**으로 재구성합니다.

- 핵심: (1) 시계열 포맷 정리, (2) 베이스라인, (3) 트렌드+계절성(푸리에 특징) + 회귀형 예측, (4) 시계열 CV 평가

의존성은 `sktime`를 중심으로 하되, 설치는 `uv`를 권장합니다.

In [ ]:
# --- Import hygiene (repo contains ./sktime source checkout) ---
# If we run from repository root, Python may resolve `import sktime` to a namespace package at ./sktime
# which breaks submodule imports. Force-import the actual package at ./sktime/sktime.
import os
import sys

repo_root = os.path.abspath(os.getcwd())
local_sktime_src = os.path.join(repo_root, "sktime")
if os.path.exists(os.path.join(local_sktime_src, "sktime", "__init__.py")):
    # remove repo root from import path (prevents namespace-package shadow)
    sys.path = [p for p in sys.path if p not in ("", repo_root)]
    # add ./sktime (so that ./sktime/sktime is the importable package)
    sys.path.insert(0, local_sktime_src)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor

from sktime.forecasting.base import ForecastingHorizon
from sktime.forecasting.model_selection import temporal_train_test_split, SlidingWindowSplitter
from sktime.forecasting.model_evaluation import evaluate
from sktime.performance_metrics.forecasting import (
    mean_absolute_error,
    mean_absolute_percentage_error,
)
from sktime.forecasting.naive import NaiveForecaster
from sktime.forecasting.ets import AutoETS
from sktime.forecasting.compose import ForecastingPipeline, RecursiveTabularRegressionForecaster
from sktime.transformations.series.fourier import FourierFeatures
from sktime.transformations.series.boxcox import LogTransformer

plt.rcParams["figure.figsize"] = (10, 4)

## 1) 데이터 로드 및 포맷 정리

기존 실습과 동일하게 `daily-website-visitors.csv`를 사용합니다.

- sktime에서는 `y`를 **Series (DatetimeIndex)** 형태로 쓰는 것이 기본입니다.

In [ ]:
path = "../datasets"
raw = pd.read_csv(os.path.join(path, "daily-website-visitors.csv"), thousands=",")
raw["Date"] = pd.to_datetime(raw["Date"])
raw = raw[["Date", "First.Time.Visits"]].rename(columns={"First.Time.Visits": "y"})
raw["y"] = pd.to_numeric(raw["y"], errors="coerce")
raw = raw.dropna().sort_values("Date")
raw = raw[raw["Date"] >= pd.to_datetime("2017-01-01")].copy()

y = raw.set_index("Date")["y"].asfreq("D")
y = y.interpolate(limit_direction="both")
y.head()

In [ ]:
y.plot(title="Daily website visitors")
plt.show()

## 2) Train/Test split + Forecasting horizon

기존 실습의 85% / 15% split을 유지합니다.

In [ ]:
y_train, y_test = temporal_train_test_split(y, test_size=int(len(y) * 0.15))
fh = ForecastingHorizon(y_test.index, is_relative=False)
len(y_train), len(y_test), fh.to_relative(cutoff=y_train.index[-1]).to_pandas().min(), fh.to_relative(cutoff=y_train.index[-1]).to_pandas().max()

## 3) 베이스라인: Naive / Seasonal Naive

Prophet/NeuralProphet 전에도 **비교 기준**을 먼저 잡는 습관이 중요합니다.

In [ ]:
baselines = {
    "Naive_last": NaiveForecaster(strategy="last"),
    "Naive_mean": NaiveForecaster(strategy="mean"),
    # weekly seasonality proxy
    "Naive_seasonal_7": NaiveForecaster(strategy="last", sp=7),
}

preds = {}
for name, f in baselines.items():
    f.fit(y_train)
    preds[name] = f.predict(fh)

ax = y_train.iloc[-120:].plot(label="train", alpha=0.7)
y_test.plot(ax=ax, label="test", alpha=0.9)
for name, yhat in preds.items():
    yhat.plot(ax=ax, label=name)
ax.legend()
ax.set_title("Baselines")
plt.show()

In [ ]:
def score(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "MAPE": float(mean_absolute_percentage_error(y_true, y_pred)),
    }

scores = {name: score(y_test, yhat) for name, yhat in preds.items()}
pd.DataFrame(scores).T.sort_values("MAPE")

## 4) Prophet/NeuralProphet에서 하던 것: 트렌드 + 계절성 + (이벤트)

여기서는 sktime에서 흔히 쓰는 방식으로 **Fourier 특징(계절성) + 회귀형 예측기(AR-like lag)**로 구성합니다.

- 계절성: `FourierFeatures(sp_list=[7, 365.25], fourier_terms_list=[3, 5])`
- 추세/스케일: 로그 변환을 파이프라인에 포함 (선택)
- AR-net 느낌: `RecursiveTabularRegressionForecaster(window_length=...)`

※ Prophet처럼 자동 changepoint를 쓰진 않지만, "구성요소를 가법적으로 결합"하는 철학은 유사합니다.

In [ ]:
forecaster = ForecastingPipeline(
    steps=[
        ("log", LogTransformer()),
        (
            "fourier",
            FourierFeatures(
                sp_list=[7, 365.25],
                fourier_terms_list=[3, 5],
                keep_original_columns=False,
            ),
        ),
        (
            "reg",
            RecursiveTabularRegressionForecaster(
                estimator=RandomForestRegressor(
                    n_estimators=300,
                    random_state=42,
                    n_jobs=-1,
                ),
                window_length=28,
            ),
        ),
    ]
)

forecaster.fit(y_train)
y_pred = forecaster.predict(fh)

ax = y_train.iloc[-180:].plot(label="train", alpha=0.7)
y_test.plot(ax=ax, label="test", alpha=0.9)
y_pred.plot(ax=ax, label="Fourier+RF(recursive)")
ax.legend()
ax.set_title("Fourier features + recursive tabular regression")
plt.show()

score(y_test, y_pred)

## 5) 시계열 Cross-Validation (sktime evaluate)

`cross_validation/performance_metrics`(기존 Prophet 방식) 대신, sktime의 splitter + evaluate를 씁니다.

In [ ]:
cv = SlidingWindowSplitter(
    fh=np.arange(1, 31),
    window_length=365,
    step_length=60,
)

results = evaluate(
    forecaster,
    y=y,
    cv=cv,
    strategy="refit",
    scoring=[mean_absolute_error, mean_absolute_percentage_error],
    return_data=False,
)
results.head()

In [ ]:
# Column names in `evaluate` output can differ slightly by sktime version.
# Select all test-metric columns automatically.
metric_cols = [c for c in results.columns if c.startswith("test_")]
results[metric_cols].describe()